In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pearsonr, spearmanr, probplot

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

df = pd.read_csv("amz_uk_processed_data.csv")  # change path if needed

print(df.shape)
display(df.head())
display(df.info())
display(df.describe(include="all").T)

In [ ]:

df.columns = df.columns.str.strip()

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0])

relevant_cols = [col for col in [
    "category", "categoryName", "isBestSeller", "price", "stars", "reviews", "boughtInLastMonth"
] if col in df.columns]

print("Relevant columns found:", relevant_cols)

In [ ]:
if "category" not in df.columns and "categoryName" in df.columns:
    df["category"] = df["categoryName"]

In [ ]:

for col in ["price", "stars", "reviews", "boughtInLastMonth"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "isBestSeller" in df.columns:
    if df["isBestSeller"].dtype != bool:
        df["isBestSeller"] = df["isBestSeller"].astype(str).str.lower().map({
            "true": True, "false": False, "1": True, "0": False
        })

display(df[["category", "isBestSeller", "price", "stars"]].head())

In [ ]:
ct = pd.crosstab(df["category"], df["isBestSeller"])
display(ct.head())

prop_best = pd.crosstab(df["category"], df["isBestSeller"], normalize="index")

if True in prop_best.columns:
    prop_best = prop_best.rename(columns={True: "best_seller_prop", False: "not_best_seller_prop"})
    best_seller_by_cat = prop_best[["best_seller_prop"]].sort_values("best_seller_prop", ascending=False)
else:
    best_seller_by_cat = pd.DataFrame(index=prop_best.index, data={"best_seller_prop": 0})

display(best_seller_by_cat.head(20))

In [ ]:
print("Top categories by proportion of best-sellers:")
display(best_seller_by_cat.head(10))

In [ ]:
chi2_table = pd.crosstab(df["category"], df["isBestSeller"])

chi2, p, dof, expected = chi2_contingency(chi2_table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")


n = chi2_table.to_numpy().sum()
r, k = chi2_table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

print(f\"Cramér's V: {cramers_v:.4f}\")

In [ ]:
if p < 0.05:
    print("Reject H0: best-seller status is not independent of category.")
else:
    print("Fail to reject H0: no evidence of association between category and best-seller status.")

print("""
Cramér's V guide:
~0.1 = weak association
~0.3 = moderate association
~0.5+ = strong association
""")

In [ ]:
top20_categories = df["category"].value_counts().head(20).index
plot_df = df[df["category"].isin(top20_categories)]

stacked = pd.crosstab(plot_df["category"], plot_df["isBestSeller"], normalize="index")
stacked = stacked.sort_values(by=True if True in stacked.columns else stacked.columns[-1], ascending=False)

stacked.plot(kind="bar", stacked=True, figsize=(14, 6))
plt.title("Best-seller proportion by category (Top 20 categories by count)")
plt.xlabel("Category")
plt.ylabel("Proportion")
plt.xticks(rotation=75)
plt.legend(title="isBestSeller")
plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.copy()

q1 = df_clean["price"].quantile(0.25)
q3 = df_clean["price"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_no_outliers = df_clean[(df_clean["price"] >= lower_bound) & (df_clean["price"] <= upper_bound)].copy()

print("Original shape:", df_clean.shape)
print("Without price outliers:", df_no_outliers.shape)
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")

In [ ]:
top20_cat_count = df_no_outliers["category"].value_counts().head(20).index
violin_df = df_no_outliers[df_no_outliers["category"].isin(top20_cat_count)]

plt.figure(figsize=(16, 7))
sns.violinplot(data=violin_df, x="category", y="price")
plt.title("Price distribution across top 20 categories")
plt.xlabel("Category")
plt.ylabel("Price")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

In [ ]:
median_price_by_cat = (
    df_no_outliers.groupby("category")["price"]
    .median()
    .sort_values(ascending=False)
)

display(median_price_by_cat.head(10))
print("Category with highest median price:", median_price_by_cat.idxmax())

In [ ]:
top10_cat_count = df_no_outliers["category"].value_counts().head(10).index
bar_df = (
    df_no_outliers[df_no_outliers["category"].isin(top10_cat_count)]
    .groupby("category")["price"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12, 6))
bar_df.plot(kind="bar")
plt.title("Average price by category (Top 10 categories by count)")
plt.xlabel("Category")
plt.ylabel("Average price")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
avg_price_by_cat = df_no_outliers.groupby("category")["price"].mean().sort_values(ascending=False)
display(avg_price_by_cat.head(10))
print("Category with highest average price:", avg_price_by_cat.idxmax())

In [ ]:
top10_cat_rating = df_no_outliers["category"].value_counts().head(10).index
rating_df = df_no_outliers[df_no_outliers["category"].isin(top10_cat_rating)]

plt.figure(figsize=(14, 6))
sns.boxplot(data=rating_df, x="category", y="stars")
plt.title("Ratings distribution by category (Top 10 categories by count)")
plt.xlabel("Category")
plt.ylabel("Stars")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
median_rating_by_cat = (
    df_no_outliers.groupby("category")["stars"]
    .median()
    .sort_values(ascending=False)
)

display(median_rating_by_cat.head(10))
print("Category with highest median rating:", median_rating_by_cat.idxmax())

In [ ]:
corr_df = df_no_outliers[["price", "stars"]].dropna()

pearson_corr, pearson_p = pearsonr(corr_df["price"], corr_df["stars"])
spearman_corr, spearman_p = spearmanr(corr_df["price"], corr_df["stars"])

print(f"Pearson correlation: {pearson_corr:.4f} | p-value: {pearson_p:.6f}")
print(f"Spearman correlation: {spearman_corr:.4f} | p-value: {spearman_p:.6f}")

In [ ]:
sample_df = df_no_outliers[["price", "stars"]].dropna().sample(
    min(10000, len(df_no_outliers)), random_state=42
)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_df, x="stars", y="price", alpha=0.4)
plt.title("Price vs. product rating")
plt.xlabel("Stars")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df_no_outliers.select_dtypes(include=np.number).columns
corr_matrix = df_no_outliers[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap of numerical variables")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
probplot(df_no_outliers["price"].dropna(), dist="norm", plot=plt)
plt.title("QQ Plot of Product Prices")
plt.tight_layout()
plt.show()

In [ ]:

median_price_by_cat_raw = df.groupby("category")["price"].median().sort_values(ascending=False)
avg_price_by_cat_raw = df.groupby("category")["price"].mean().sort_values(ascending=False)

display(median_price_by_cat_raw.head(10))
display(avg_price_by_cat_raw.head(10))